In [1]:
import os
import shutil

project_root = os.getcwd()
for root, dirs, files in os.walk(project_root):
    if '__pycache__' in dirs:
        pycache_dir = os.path.join(root, '__pycache__')
        print(f"Deleting: {pycache_dir}")
        shutil.rmtree(pycache_dir)

print("--- All __pycache__ directories deleted. ---")

Deleting: /Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/__pycache__
--- All __pycache__ directories deleted. ---


In [1]:
from scripts import models
import sys

# This will tell us WHICH models.py file is being loaded
print(f"Importing models from: {models.__file__}")

# This will show us where Python is looking for files
print("\nPython Path:")
print('\n'.join(sys.path))

Importing models from: /Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/models.py

Python Path:
/opt/anaconda3/envs/BTP/lib/python310.zip
/opt/anaconda3/envs/BTP/lib/python3.10
/opt/anaconda3/envs/BTP/lib/python3.10/lib-dynload

/opt/anaconda3/envs/BTP/lib/python3.10/site-packages
/var/folders/dw/8xmb3tsd1j97gg1p6nfz7vk00000gn/T/tmp91kjsvok


In [2]:
#
# Cell 1: Train GraphSAGE Teacher Model (on 20% subset)
#
import yaml
from pathlib import Path
import os
import torch
from torch_geometric.nn import to_hetero
import torch_geometric.transforms as T
from scripts import models

# --- Configuration ---
DATASET_NAME = 'ogbnarxiv'
SUBSET_RATIO = 0.20
TEACHER_MODEL = 'SAGE'

print(f"--- Phase 1: Training the {TEACHER_MODEL} Teacher Model on {int(SUBSET_RATIO*100)}% subset ---")

# --- Define paths ---
project_root = Path.cwd()
subset_data_path = project_root / 'data' / f"{DATASET_NAME}_heterodata_subset_{int(SUBSET_RATIO*100)}pct.pt"

if not subset_data_path.exists():
    raise FileNotFoundError(f"Subset file not found at {subset_data_path}")

# --- 1. Prepare teacher config (same as your working original benchmark) ---
exp_params = {
    'dataset': DATASET_NAME,
    'data': {'graph_dataset': {DATASET_NAME: str(subset_data_path)}},
    'node_embs': 'preloaded_in_subset',
    'edge_type_selection': {DATASET_NAME: ['references', 'author', 'venue', 'fos']},
    'optimizer': {'lr': 0.001, 'weight_decay': 0},
    'model': {
        'name': TEACHER_MODEL,
        'hidden_channels': 128,
        'num_layers': 2,
        'dropout': 0.5
    },
    'early_stop_threshold': 30,
    'epochs': 500,      # ✅ Added to fix KeyError
    'runs': 1           # Only 1 run for generating pseudo-labels
}

# --- 2. Save temp config ---
temp_config_path = f"config/temp_{TEACHER_MODEL.lower()}_teacher_20pct.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(exp_params, f, sort_keys=False)

# --- 3. Train Teacher ---
print(f"\nRunning training for {TEACHER_MODEL} teacher...")
%run scripts/experiments.py --config {temp_config_path}
print("Teacher training finished.")

# --- 4. Generate and Save Pseudo-labels ---
print("\nGenerating pseudo-labels from the trained SAGE teacher...")
teacher_data = torch.load(str(subset_data_path))
teacher_data = T.ToSparseTensor()(teacher_data)

in_channels = teacher_data['paper'].x.shape[1]
out_channels = len(torch.unique(teacher_data['paper'].y))

teacher_model = models.SAGE(
    num_layers=2,
    in_channels=in_channels,
    out_channels=out_channels,
    hidden_channels=128,
    dropout=0.5
)
teacher_model = to_hetero(teacher_model, teacher_data.metadata(), aggr='mean')

teacher_weights_path = f"models/{DATASET_NAME}_{TEACHER_MODEL}_run0_best.pth"
teacher_model.load_state_dict(torch.load(teacher_weights_path))
teacher_model.eval()

with torch.no_grad():
    pseudo_labels = teacher_model(teacher_data.x_dict, teacher_data.adj_t_dict)['paper']

pseudo_labels_path = "data/embeddings/sage_teacher_predictions_20pct.pt"
torch.save(pseudo_labels.cpu(), pseudo_labels_path)
print(f"SAGE pseudo-labels saved to: {pseudo_labels_path}")

--- Phase 1: Training the SAGE Teacher Model on 20% subset ---

Running training for SAGE teacher...
Will run on: cpu.
Converting edge_index to SparseTensor format (adj_t)...


/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:184: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path_to_dat

Conversion complete.
Using node features pre-loaded in the data object. Shape: torch.Size([33868, 1024])
Model: SAGE
No. parameters:  1090464


Run 00:   0%|                                           | 0/500 [00:00<?, ?it/s]/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_sparse/tensor.py:574: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:55.)
  return torch.sparse_csr_tensor(rowptr, col, value, self.sizes())
Run 00:  32%|██████████▌                      | 160/500 [05:22<11:25,  2.02s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:250: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/b

Early stopped training for run 00 at epoch 161.
Run 00: Best Epoch 130, Best Val Acc 0.7629, Test Acc 0.7728.

* ============================= ALL RUNS SUMMARY =============================
Best Val Acc: 76.29, Corresponding Test Acc: 77.28
Avg. Test Acc: 77.28 ± nan.
* ===========================================================================
Teacher training finished.

Generating pseudo-labels from the trained SAGE teacher...


/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:260: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1823.)
  print(f"Avg. Test Acc: {all_runs_accs[:, 1].mean().item()*100:.2f} ± {all_runs_accs[:, 1].std().item()*100:.2f}.")
/var/folders/dw/8xmb3tsd1j97gg1p6nfz7vk00000gn/T/ipykernel_20742/188221482.py:56: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits th

SAGE pseudo-labels saved to: data/embeddings/sage_teacher_predictions_20pct.pt


In [3]:
#
# Cell 2: Train RevGAT Student (Distillation from SAGE Teacher)
#
import yaml
from pathlib import Path
import os

# --- Configuration ---
DATASET_NAME = 'ogbnarxiv'
STUDENT_MODEL = 'RevGAT'
SUBSET_RATIO = 0.20

print(f"\n--- Phase 2: Training {STUDENT_MODEL} Student using SAGE Teacher (20% subset) ---")

# --- Define paths ---
project_root = Path.cwd()
subset_data_path = project_root / 'data' / f"{DATASET_NAME}_heterodata_subset_{int(SUBSET_RATIO*100)}pct.pt"
pseudo_labels_path = project_root / 'data' / 'embeddings' / 'sage_teacher_predictions_20pct.pt'

if not subset_data_path.exists():
    raise FileNotFoundError(f"Subset file not found: {subset_data_path}")
if not pseudo_labels_path.exists():
    raise FileNotFoundError(f"Pseudo-labels not found: {pseudo_labels_path}")

# --- 1. Prepare student config ---
exp_params = {
    'dataset': DATASET_NAME,
    'data': {'graph_dataset': {DATASET_NAME: str(subset_data_path)}},
    'node_embs': 'preloaded_in_subset',
    'edge_type_selection': {DATASET_NAME: ['references', 'author', 'venue', 'fos']},
    'augment_with_preds': str(pseudo_labels_path), # <-- Still using the teacher's labels
    'optimizer': {'lr': 0.001, 'weight_decay': 0.0005},
    'model': {
        'name': STUDENT_MODEL,
        'hidden_channels': 64,  # <-- 1. REDUCED from 128
        'num_layers': 2,
        'dropout': 0.5,
        'heads': 4               # <-- You could also try reducing this to 2
    },
    'early_stop_threshold': 30,
    'epochs': 500,
    'runs': 3
}
# --- 2. Save temp config ---
temp_config_path = f"config/temp_{STUDENT_MODEL.lower()}_student_sage_teacher_20pct.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(exp_params, f, sort_keys=False)

# --- 3. Train Student ---
print(f"Running training for {STUDENT_MODEL} (SAGE teacher distillation)...")
%run scripts/experiments.py --config {temp_config_path}
print("\nStudent training finished successfully.")


--- Phase 2: Training RevGAT Student using SAGE Teacher (20% subset) ---
Running training for RevGAT (SAGE teacher distillation)...
Will run on: cpu.
Converting edge_index to SparseTensor format (adj_t)...


/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:184: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path_to_dat

Conversion complete.
Using node features pre-loaded in the data object. Shape: torch.Size([33868, 1024])
Model: RevGAT
No. parameters:  423200


Run 00:  36%|████████████                     | 182/500 [26:21<46:02,  8.69s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:250: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 00 at epoch 183.


...

Early stopped training for run 01 at epoch 208.


/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "
/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/to_hetero_transformer.py:379: UserWarning: 'rev_blocks.0' will be duplicated, but its parameters cannot be reset. To suppress this warning, add a 'reset_parameters()' method to 'rev_blocks.0'
  warnings.warn(
/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/to_hetero_transformer.py:379: UserWarning: 'rev_blocks.1' will be duplicated, but its parameters cannot be reset. To suppress this warning, add a 'reset_parameters()' method to 'rev_blocks.1'
  warnings.warn(


Run 01: Best Epoch 177, Best Val Acc 0.7635, Test Acc 0.7616.


Run 02:  34%|███████████▏                     | 169/500 [24:59<48:57,  8.88s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:250: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 02 at epoch 170.
Run 02: Best Epoch 139, Best Val Acc 0.7579, Test Acc 0.7672.

* ============================= ALL RUNS SUMMARY =============================
Best Val Acc: 76.35, Corresponding Test Acc: 76.16
Avg. Test Acc: 76.47 ± 0.29.
* ===========================================================================

Student training finished successfully.


In [4]:
import yaml
from pathlib import Path
import os
import torch
from torch_geometric.nn import to_hetero
import torch_geometric.transforms as T
import models

# --- Configuration ---
DATASET_NAME = 'ogbnarxiv'
SUBSET_RATIO = 0.40  # <-- CHANGED TO 40%
TEACHER_MODEL = 'SAGE'

print(f"--- Phase 1 (40%): Training the {TEACHER_MODEL} Teacher Model on {int(SUBSET_RATIO*100)}% subset ---")

# --- Define paths ---
project_root = Path.cwd()
subset_data_path = project_root / 'data' / f"{DATASET_NAME}_heterodata_subset_{int(SUBSET_RATIO*100)}pct.pt"
PSEUDO_LABELS_PATH_40 = "data/embeddings/sage_teacher_predictions_40pct.pt" # <-- NEW PATH

if not subset_data_path.exists():
    raise FileNotFoundError(f"Subset file not found at {subset_data_path}")

# --- 1. Prepare teacher config ---
exp_params = {
    'dataset': DATASET_NAME,
    'data': {'graph_dataset': {DATASET_NAME: str(subset_data_path)}},
    'node_embs': 'preloaded_in_subset',
    'edge_type_selection': {DATASET_NAME: ['references', 'author', 'venue', 'fos']},
    'optimizer': {'lr': 0.001, 'weight_decay': 0},
    'model': {
        'name': TEACHER_MODEL,
        'hidden_channels': 128,
        'num_layers': 2,
        'dropout': 0.5
    },
    'early_stop_threshold': 30,
    'epochs': 500,
    'runs': 1  # Only 1 run for generating pseudo-labels
}

# --- 2. Save temp config ---
temp_config_path = f"config/temp_{TEACHER_MODEL.lower()}_teacher_40pct.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(exp_params, f, sort_keys=False)

# --- 3. Train Teacher ---
print(f"\nRunning training for {TEACHER_MODEL} teacher...")
%run scripts/experiments.py --config {temp_config_path}
print("Teacher training finished.")

# --- 4. Generate and Save Pseudo-labels ---
print("\nGenerating pseudo-labels from the trained SAGE teacher...")
teacher_data = torch.load(str(subset_data_path))
teacher_data = T.ToSparseTensor()(teacher_data)

in_channels = teacher_data['paper'].x.shape[1]
out_channels = len(torch.unique(teacher_data['paper'].y))

teacher_model = models.SAGE(
    num_layers=2,
    in_channels=in_channels,
    out_channels=out_channels,
    hidden_channels=128,
    dropout=0.5
)
# Note: Ensure your 'models.SAGE' definition works with the to_hetero wrapper
teacher_model = to_hetero(teacher_model, teacher_data.metadata(), aggr='mean')

teacher_weights_path = f"models/{DATASET_NAME}_{TEACHER_MODEL}_run0_best.pth"
teacher_model.load_state_dict(torch.load(teacher_weights_path))
teacher_model.eval()

with torch.no_grad():
    # This might return a dict or tensor depending on your model/wrapper. 
    # Based on your previous runs, it seems to return a dict keyed by 'paper'.
    out_raw = teacher_model(teacher_data.x_dict, teacher_data.adj_t_dict)
    
    if isinstance(out_raw, dict) and 'paper' in out_raw:
        pseudo_labels = out_raw['paper']
    elif torch.is_tensor(out_raw):
        pseudo_labels = out_raw
    # If the output is the canonical edge type dict (as seen in RevGAT), 
    # this manual script would need the aggregation logic from the fix I provided earlier,
    # but we'll stick to the working SAGE logic from your 20% run for simplicity.
    else:
        # Fallback to the working logic for SAGE from your 20% run
        pseudo_labels = teacher_model(teacher_data.x_dict, teacher_data.adj_t_dict)['paper']


torch.save(pseudo_labels.cpu(), PSEUDO_LABELS_PATH_40)
print(f"SAGE pseudo-labels saved to: {PSEUDO_LABELS_PATH_40}")

--- Phase 1 (40%): Training the SAGE Teacher Model on 40% subset ---

Running training for SAGE teacher...
Will run on: cpu.
Converting edge_index to SparseTensor format (adj_t)...


/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:184: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path_to_dat

Conversion complete.
Using node features pre-loaded in the data object. Shape: torch.Size([67737, 1024])
Model: SAGE
No. parameters:  1090464


Run 00:  34%|███████████▏                     | 169/500 [07:11<14:04,  2.55s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:250: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 00 at epoch 170.
Run 00: Best Epoch 139, Best Val Acc 0.7694, Test Acc 0.7668.

* ============================= ALL RUNS SUMMARY =============================
Best Val Acc: 76.94, Corresponding Test Acc: 76.68
Avg. Test Acc: 76.68 ± nan.
* ===========================================================================
Teacher training finished.

Generating pseudo-labels from the trained SAGE teacher...


/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:260: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1823.)
  print(f"Avg. Test Acc: {all_runs_accs[:, 1].mean().item()*100:.2f} ± {all_runs_accs[:, 1].std().item()*100:.2f}.")
/var/folders/dw/8xmb3tsd1j97gg1p6nfz7vk00000gn/T/ipykernel_20742/1118444571.py:54: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits t

SAGE pseudo-labels saved to: data/embeddings/sage_teacher_predictions_40pct.pt


In [6]:
import yaml
from pathlib import Path
import os

# --- Configuration ---
DATASET_NAME = 'ogbnarxiv'
STUDENT_MODEL = 'RevGAT'
SUBSET_RATIO = 0.40 # <-- CHANGED TO 40%

print(f"\n--- Phase 2 (40%): Training {STUDENT_MODEL} Student using SAGE Teacher ---")

# --- Define paths ---
project_root = Path.cwd()
subset_data_path = project_root / 'data' / f"{DATASET_NAME}_heterodata_subset_{int(SUBSET_RATIO*100)}pct.pt"
pseudo_labels_path = project_root / 'data' / 'embeddings' / 'sage_teacher_predictions_40pct.pt' # <-- NEW PATH

if not subset_data_path.exists():
    raise FileNotFoundError(f"Subset file not found: {subset_data_path}")
if not pseudo_labels_path.exists():
    raise FileNotFoundError(f"Pseudo-labels not found: {pseudo_labels_path}")

# --- 1. Prepare student config ---
exp_params = {
    'dataset': DATASET_NAME,
    'data': {'graph_dataset': {DATASET_NAME: str(subset_data_path)}},
    'node_embs': 'preloaded_in_subset',
    'edge_type_selection': {DATASET_NAME: ['references', 'author', 'venue', 'fos']},
    'augment_with_preds': str(pseudo_labels_path), # <-- Still using the teacher's labels
    'optimizer': {'lr': 0.001, 'weight_decay': 0.0005},
    'model': {
        'name': STUDENT_MODEL,
        'hidden_channels': 64,  # <-- 1. REDUCED from 128
        'num_layers': 2,
        'dropout': 0.5,
        'heads': 2              # <-- You could also try reducing this to 2
    },
    'early_stop_threshold': 30,
    'epochs': 500,
    'runs': 3
}

# --- 2. Save temp config ---
temp_config_path = f"config/temp_{STUDENT_MODEL.lower()}_student_sage_teacher_40pct.yml"
with open(temp_config_path, 'w') as f:
    yaml.dump(exp_params, f, sort_keys=False)

# --- 3. Train Student ---
print(f"Running training for {STUDENT_MODEL} (SAGE teacher distillation)...")
# Note: This run requires your corrected scripts/experiments.py file
%run scripts/experiments.py --config {temp_config_path}
print("\nStudent training finished successfully.")


--- Phase 2 (40%): Training RevGAT Student using SAGE Teacher ---
Running training for RevGAT (SAGE teacher distillation)...
Will run on: cpu.


/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:184: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(path_to_dat

Converting edge_index to SparseTensor format (adj_t)...


/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "
/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/to_hetero_transformer.py:379: UserWarning: 'rev_blocks.0' will be duplicated, but its parameters cannot be reset. To suppress this warning, add a 'reset_parameters()' method to 'rev_blocks.0'
  warnings.warn(
/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/to_hetero_transformer.py:379: UserWarning: 'rev_blocks.1' will be duplicated, but its parameters cannot be reset. To suppress this warning, add a 'reset_parameters()' method to 'rev_blocks.1'
  warnings.warn(


Conversion complete.
Using node features pre-loaded in the data object. Shape: torch.Size([67737, 1024])
Model: RevGAT
No. parameters:  288032


...

Early stopped training for run 01 at epoch 256.


/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/fx.py:132: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  warnings.warn(f"Found function '{node.name}' with keyword "
/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/to_hetero_transformer.py:379: UserWarning: 'rev_blocks.0' will be duplicated, but its parameters cannot be reset. To suppress this warning, add a 'reset_parameters()' method to 'rev_blocks.0'
  warnings.warn(
/opt/anaconda3/envs/BTP/lib/python3.10/site-packages/torch_geometric/nn/to_hetero_transformer.py:379: UserWarning: 'rev_blocks.1' will be duplicated, but its parameters cannot be reset. To suppress this warning, add a 'reset_parameters()' method to 'rev_blocks.1'
  warnings.warn(


Run 01: Best Epoch 225, Best Val Acc 0.7711, Test Acc 0.7653.


Run 02:  36%|██████████▍                  | 181/500 [1:13:18<2:09:11, 24.30s/it]
/Users/macsaumya/Documents/BTP/Model and Data/edgehetero-nodeproppred/scripts/experiments.py:250: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for

Early stopped training for run 02 at epoch 182.
Run 02: Best Epoch 151, Best Val Acc 0.7688, Test Acc 0.7650.

* ============================= ALL RUNS SUMMARY =============================
Best Val Acc: 77.11, Corresponding Test Acc: 76.53
Avg. Test Acc: 76.41 ± 0.18.
* ===========================================================================

Student training finished successfully.
